In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__output__.json
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__results__.html
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/custom.css
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_val.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_val.csv


In [31]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)
(28831,)
(6178,)
(6179,)


In [32]:
X_train.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,50,blue-collar,single,basic.4y,no,no,no,telephone,may,thu,2,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0
1,51,services,married,professional.course,unknown,no,no,telephone,may,mon,5,999,0,nonexistent,1.1,93.994,-36.4,4.858,5191.0
2,46,technician,divorced,professional.course,no,yes,no,cellular,may,mon,2,999,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1
3,46,admin.,single,university.degree,no,yes,no,telephone,jun,mon,1,999,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1
4,25,services,single,high.school,no,no,no,cellular,may,thu,5,999,0,nonexistent,-1.8,92.893,-46.2,1.266,5099.1


In [33]:
v2_drop=["emp.var.rate", "euribor3m"]
X_train.drop(columns=v2_drop, inplace=True)
X_val.drop(columns=v2_drop, inplace=True)
X_test.drop(columns=v2_drop, inplace=True)

X_train["pdays_flag"] = (X_train["pdays"] != 999).astype(int)
X_train["previos_flag"] = (X_train["previous"] > 0).astype(int)

X_val["pdays_flag"] = (X_val["pdays"] != 999).astype(int)
X_val["previos_flag"] = (X_val["previous"] > 0).astype(int)

X_test["pdays_flag"] = (X_test["pdays"] != 999).astype(int)
X_test["previos_flag"] = (X_test["previous"] > 0).astype(int)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)


In [34]:
def calculate_woe_iv(X, y, max_bins=10):
    
    data = X.copy()
    data["target"] = np.array(y)
    
    iv_summary = []
    woe_tables = []
    
    for feature in X.columns:
        
        temp = data[[feature, "target"]].copy()
        
        # --------------------------------------------------
        # 1. Create bins for numerical variables
        # --------------------------------------------------
        if pd.api.types.is_numeric_dtype(temp[feature]):
            
            try:
                temp["bin"] = pd.qcut(
                    temp[feature],
                    q=max_bins,
                    duplicates="drop"
                )
            except:
                temp["bin"] = temp[feature].astype(str)
        
        # --------------------------------------------------
        # 2. Keep categories as they are
        # --------------------------------------------------
        else:
            temp["bin"] = temp[feature].fillna("Missing")
        
        # --------------------------------------------------
        # 3. Calculate counts
        # --------------------------------------------------
        summary = (
            temp.groupby("bin", observed=False)["target"]
            .agg(
                Total="size",
                Events="sum"
            )
            .reset_index()
        )
        
        summary["Non-Events"] = (
            summary["Total"] - summary["Events"]
        )
        
        # --------------------------------------------------
        # 4. Distribution of events/non-events
        # --------------------------------------------------
        total_events = summary["Events"].sum()
        total_non_events = summary["Non-Events"].sum()
        
        # Small smoothing to avoid division by zero
        n_bins = len(summary)
        
        summary["Event Distribution"] = (
            (summary["Events"] + 0.5) /
            (total_events + 0.5 * n_bins)
        )
        
        summary["Non-Event Distribution"] = (
            (summary["Non-Events"] + 0.5) /
            (total_non_events + 0.5 * n_bins)
        )
        
        # --------------------------------------------------
        # 5. Weight of Evidence
        # Convention:
        # WoE = ln(Non-event distribution / Event distribution)
        # --------------------------------------------------
        summary["WoE"] = np.log(
            summary["Non-Event Distribution"] /
            summary["Event Distribution"]
        )
        
        # --------------------------------------------------
        # 6. Information Value
        # --------------------------------------------------
        summary["IV"] = (
            summary["Non-Event Distribution"] -
            summary["Event Distribution"]
        ) * summary["WoE"]
        
        feature_iv = summary["IV"].sum()
        
        # --------------------------------------------------
        # 7. Store IV summary
        # --------------------------------------------------
        iv_summary.append({
            "Feature": feature,
            "IV": feature_iv
        })
        
        # --------------------------------------------------
        # 8. Store detailed WoE table
        # --------------------------------------------------
        summary.insert(0, "Feature", feature)
        
        woe_tables.append(summary)
    
    iv_report = (
        pd.DataFrame(iv_summary)
        .sort_values("IV", ascending=False)
        .reset_index(drop=True)
    )
    
    woe_report = pd.concat(
        woe_tables,
        ignore_index=True
    )
    
    return iv_report, woe_report

In [35]:
iv_report, woe_report = calculate_woe_iv(
    X_train,
    y_train,
    max_bins=10
)
iv_report.round(4)

,Feature,IV
0,nr.employed,1.1363
1,cons.conf.idx,0.8341
2,poutcome,0.5474
3,month,0.4728
4,cons.price.idx,0.4136
5,contact,0.2431
6,previous,0.2172
7,job,0.1772
8,age,0.1384
9,default,0.1225


In [21]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 52)
Validation: (6178, 52)
Test: (6179, 52)


In [22]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [23]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 200
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 7
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 0.1, 10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10,
            log=True
        )
    }

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        tree_method="hist"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return scores.mean()

In [24]:
os.makedirs("/kaggle/working/outputs", exist_ok=True)

log_path = "/kaggle/working/outputs/optuna_xgb_trials_07.csv"

def log_optuna_trial(study, trial):

    rows = []

    for t in study.trials:

        row = {
            "Trial": t.number,
            "Current CV ROC-AUC": t.value,
            "Best CV ROC-AUC": study.best_value,
            "Best Trial": study.best_trial.number,
            "Is Best": t.number == study.best_trial.number
        }

        row.update(t.params)
        rows.append(row)

    pd.DataFrame(rows).sort_values("Trial").to_csv(
        log_path,
        index=False
    )

In [25]:
!pip install -q optuna
import optuna
study = optuna.create_study(
    direction="maximize",
    study_name="xgboost_roc_auc"
)

study.optimize(
    objective,
    n_trials=150,
    callbacks=[log_optuna_trial]
)

[I 2026-11-23 08:08:29,090] A new study created in memory with name: xgboost_roc_auc
[I 2026-11-23 08:08:33,553] Trial 0 finished with value: 0.7907793223909337 and parameters: {'n_estimators': 198, 'learning_rate': 0.015195305041536149, 'max_depth': 6, 'min_child_weight': 9, 'subsample': 0.7986677536356609, 'colsample_bytree': 0.9847034299100463, 'gamma': 3.727181005033824, 'reg_alpha': 1.16478913865196, 'reg_lambda': 0.0038161417325469973}. Best is trial 0 with value: 0.7907793223909337.
[I 2026-11-23 08:08:35,166] Trial 1 finished with value: 0.7913349404184694 and parameters: {'n_estimators': 119, 'learning_rate': 0.02734208066150503, 'max_depth': 6, 'min_child_weight': 9, 'subsample': 0.884236017906394, 'colsample_bytree': 0.7396875682365619, 'gamma': 1.6324001942323778, 'reg_alpha': 2.8693167797368075, 'reg_lambda': 0.0012698765844849913}. Best is trial 1 with value: 0.7913349404184694.
[I 2026-11-23 08:08:36,827] Trial 2 finished with value: 0.7917862144622718 and parameters: {'

In [26]:
optuna_results = pd.read_csv(
    "/kaggle/working/outputs/optuna_xgb_trials_07.csv"
)

optuna_results.round(4)

,Trial,Current CV ROC-AUC,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,0,0.7908,0.7945,116,False,198,0.0152,6,9,0.7987,0.9847,3.7272,1.1648,0.0038
1,1,0.7913,0.7945,116,False,119,0.0273,6,9,0.8842,0.7397,1.6324,2.8693,0.0013
2,2,0.7918,0.7945,116,False,200,0.0509,3,1,0.9319,0.6551,0.4026,0.1363,0.1632
3,3,0.7911,0.7945,116,False,88,0.0108,6,1,0.6795,0.7227,3.6080,0.5406,1.2494
4,4,0.7892,0.7945,116,False,115,0.0635,3,5,0.9557,0.9377,3.5130,0.5967,5.0137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,145,0.7939,0.7945,116,False,172,0.0305,7,4,0.9084,0.6490,2.6016,0.7543,0.0346
146,146,0.7943,0.7945,116,False,189,0.0342,7,4,0.9449,0.6569,2.5748,0.5758,0.0740
147,147,0.7935,0.7945,116,False,193,0.0410,7,4,0.9470,0.6515,2.4337,0.7333,0.0429
148,148,0.7943,0.7945,116,False,183,0.0387,7,3,0.9319,0.6579,2.6593,0.3689,0.0234


In [27]:
print("Best Trial:", study.best_trial.number)
print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best Trial: 116
Best CV ROC-AUC: 0.7945417344201063
Best Parameters:
n_estimators: 182
learning_rate: 0.03303605145015818
max_depth: 7
min_child_weight: 3
subsample: 0.9528784247282855
colsample_bytree: 0.6793523290987519
gamma: 2.335018456453509
reg_alpha: 0.4824726607783658
reg_lambda: 0.03493191023825116


In [28]:
best_xgb = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist"
)

best_xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_xgb)
])

best_xgb_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'cons.price.idx',
                                                   'cons.conf.idx',
                                                   'nr.employed', 'pdays_flag',
                                                   'previos_flag']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(dro...
                               gamma=2.335018456453509, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.03303605145015818, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=7,
                               max_leaves=None, min_child_weight=3, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=182, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_val_pred = best_xgb_pipeline.predict(X_val)
y_val_prob = best_xgb_pipeline.predict_proba(X_val)[:, 1]

tuned_xgb_results = {
    "Model": "XGBoost Tuned",
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

pd.DataFrame([tuned_xgb_results]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost Tuned,0.9021,0.6813,0.2457,0.3611,0.808
